In [12]:
import pandas as pd
import numpy as np
import json

# --- INGESTION ---
try:
    with open('../data/raw_credit_applications.json', 'r') as f:
        df = pd.json_normalize(json.load(f))
    print(f"[SYSTEM] Logged: {len(df)} records ingested.")
except Exception as e:
    print(f"[ERROR] Ingestion failed: {e}")

# --- DATA QUALITY REPORT ---
def generate_dq_report(df):
    print("\n" + "="*60)
    print(" NOVACRED DATA QUALITY REPORT v1.0")
    print("="*60)
    
    # 1. Duplicate Records
    dupes = df.duplicated(subset=['_id']).sum()
    print(f"I.   UNIQUENESS: {dupes} duplicate(s) identified ({(dupes/len(df)*100):.2f}%)")

    # 2. Inconsistent Data Types
    print("\nII.  CONSISTENCY (DATA TYPES):")
    if 'financials.annual_income' in df.columns:
        types = df['financials.annual_income'].apply(type).value_counts()
        for dtype, count in types.items():
            print(f"     - financials.annual_income | {dtype}: {count}")

    # 3. Missing or Incomplete Records
    print("\nIII. COMPLETENESS (TOP MISSING FIELDS):")
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df)) * 100
    missing_report = pd.DataFrame({'counts': null_counts, 'pct': null_pct})
    top_missing = missing_report[missing_report['counts'] > 0].sort_values(by='pct', ascending=False)
    for field, row in top_missing.iterrows():
        print(f"     - {field:30} | {int(row['counts']):3} missing | {row['pct']:5.1f}%")

    # 4. Inconsistent Coding/Formatting (Categorical)
    print("\nIV.  CONSISTENCY (CATEGORICAL CODING):")
    if 'applicant_info.gender' in df.columns:
        unique_vals = [str(x) for x in df['applicant_info.gender'].unique()]
        print(f"     - applicant_info.gender | Observed: {', '.join(unique_vals)}")

    # 5. Invalid or Impossible Values
    print("\nV.   VALIDITY (DOMAIN CONSTRAINTS):")
    if 'financials.credit_history_months' in df.columns:
        invalid = (df['financials.credit_history_months'] < 0).sum()
        print(f"     - financials.credit_history_months | Negatives detected: {invalid}")

    # 6. Inconsistent Date Formats
    print("\nVI.  ACCURACY (TEMPORAL FORMATS):")
    if 'applicant_info.date_of_birth' in df.columns:
        samples = df['applicant_info.date_of_birth'].head(8).to_list()
        print(f"     - applicant_info.date_of_birth | Samples: {samples}")
    
    print("="*60)

generate_dq_report(df)

[SYSTEM] Logged: 502 records ingested.

 NOVACRED DATA QUALITY REPORT v1.0
I.   UNIQUENESS: 2 duplicate(s) identified (0.40%)

II.  CONSISTENCY (DATA TYPES):
     - financials.annual_income | <class 'int'>: 488
     - financials.annual_income | <class 'str'>: 8
     - financials.annual_income | <class 'float'>: 6

III. COMPLETENESS (TOP MISSING FIELDS):
     - notes                          | 500 missing |  99.6%
     - financials.annual_salary       | 497 missing |  99.0%
     - loan_purpose                   | 452 missing |  90.0%
     - processing_timestamp           | 440 missing |  87.6%
     - decision.rejection_reason      | 292 missing |  58.2%
     - decision.interest_rate         | 210 missing |  41.8%
     - decision.approved_amount       | 210 missing |  41.8%
     - applicant_info.ssn             |   5 missing |   1.0%
     - applicant_info.ip_address      |   5 missing |   1.0%
     - financials.annual_income       |   5 missing |   1.0%
     - applicant_info.gender      

In [13]:
# Create a copy for cleaning
df_clean = df.copy()

# 1. FIXING DATES: Convert all to a standard datetime object
# 'dayfirst=True' helps with the DD/MM/YYYY records you found
df_clean['applicant_info.date_of_birth'] = pd.to_datetime(df_clean['applicant_info.date_of_birth'], dayfirst=True, errors='coerce')

# 2. FIXING INCOME: Convert all to numeric (floats)
# This handles the strings and integers you found in the audit
df_clean['financials.annual_income'] = pd.to_numeric(df_clean['financials.annual_income'], errors='coerce')

# 3. FIXING GENDER: Standardize the messy coding
gender_map = {
    'Male': 'Male', 'M': 'Male', 'm': 'Male',
    'Female': 'Female', 'F': 'Female', 'f': 'Female',
    '': np.nan # Treat empty strings as missing
}
df_clean['applicant_info.gender'] = df_clean['applicant_info.gender'].replace(gender_map)

# 4. FIXING VALIDITY: Remove negative credit months
# Setting negative values to NaN or 0 (decide based on team discussion)
df_clean.loc[df_clean['financials.credit_history_months'] < 0, 'financials.credit_history_months'] = np.nan

# 5. REMOVING DUPLICATES
df_clean = df_clean.drop_duplicates(subset=['_id'])

print(f"✅ Cleaning complete. Records remaining: {len(df_clean)}")

✅ Cleaning complete. Records remaining: 500


In [14]:
# --- POST-REMEDIATION VERIFICATION REPORT ---

def generate_verification_report(df_new):
    print("\n" + "="*60)
    print(" NOVACRED DATA QUALITY VERIFICATION (POST-CLEANING)")
    print("="*60)
    
    # I. Duplicate Records
    dupes = df_new.duplicated(subset=['_id']).sum()
    status_i = "PASSED" if dupes == 0 else "FAILED"
    print(f"I.   UNIQUENESS: {dupes} duplicates remaining | STATUS: {status_i}")

    # II. Inconsistent Data Types
    print("\nII.  CONSISTENCY (DATA TYPES):")
    if 'financials.annual_income' in df_new.columns:
        types = df_new['financials.annual_income'].apply(type).value_counts()
        is_float_only = all(t == type(0.0) for t in types.index)
        status_ii = "PASSED" if is_float_only else "FAILED"
        for dtype, count in types.items():
            print(f"     - financials.annual_income | {dtype}: {count}")
        print(f"     STATUS: {status_ii}")

    # III. Missing or Incomplete Records
    print("\nIII. COMPLETENESS (REMAINING NULLS):")
    null_total = df_new.isnull().sum().sum()
    # Note: Some nulls are expected (e.g., rejection_reason for approved loans)
    print(f"     - Total null values across dataset: {null_total}")
    print(f"     - Record count: {len(df_new)}")

    # IV. Inconsistent Coding/Formatting (Categorical)
    print("\nIV.  CONSISTENCY (CATEGORICAL CODING):")
    if 'applicant_info.gender' in df_new.columns:
        unique_vals = df_new['applicant_info.gender'].unique().tolist()
        # Logic check: Should only contain ['Male', 'Female', nan]
        expected = {'Male', 'Female', np.nan}
        is_standardized = set(unique_vals).issubset(expected)
        status_iv = "PASSED" if is_standardized else "FAILED"
        print(f"     - applicant_info.gender | Observed: {unique_vals}")
        print(f"     STATUS: {status_iv}")

    # V. Invalid or Impossible Values
    print("\nV.   VALIDITY (DOMAIN CONSTRAINTS):")
    if 'financials.credit_history_months' in df_new.columns:
        invalid = (df_new['financials.credit_history_months'] < 0).sum()
        status_v = "PASSED" if invalid == 0 else "FAILED"
        print(f"     - financials.credit_history_months | Negatives: {invalid}")
        print(f"     STATUS: {status_v}")

    # VI. Inconsistent Date Formats
    print("\nVI.  ACCURACY (TEMPORAL FORMATS):")
    is_dt = pd.api.types.is_datetime64_any_dtype(df_new['applicant_info.date_of_birth'])
    status_vi = "PASSED" if is_dt else "FAILED"
    print(f"     - applicant_info.date_of_birth | Datetime Object: {is_dt}")
    print(f"     STATUS: {status_vi}")
    
    print("="*60)

# Run the verification on your cleaned dataframe
generate_verification_report(df_clean)


 NOVACRED DATA QUALITY VERIFICATION (POST-CLEANING)
I.   UNIQUENESS: 0 duplicates remaining | STATUS: PASSED

II.  CONSISTENCY (DATA TYPES):
     - financials.annual_income | <class 'float'>: 500
     STATUS: PASSED

III. COMPLETENESS (REMAINING NULLS):
     - Total null values across dataset: 2965
     - Record count: 500

IV.  CONSISTENCY (CATEGORICAL CODING):
     - applicant_info.gender | Observed: ['Male', 'Female', nan]
     STATUS: PASSED

V.   VALIDITY (DOMAIN CONSTRAINTS):
     - financials.credit_history_months | Negatives: 0
     STATUS: PASSED

VI.  ACCURACY (TEMPORAL FORMATS):
     - applicant_info.date_of_birth | Datetime Object: True
     STATUS: PASSED
